# 面试问题：多个大模型之间怎样做质量、成本和延迟感知的动态路由？

**一句话回答**：先定义每请求质量/风险/SLO 和各模型成本能力，离线让候选模型全部跑同一日志得到 counterfactual 标签；学习“升级到强模型的边际收益”，再在质量与风险硬门槛下优化成本/延迟。线上保留置信拒绝、强制高风险路由、fallback/circuit breaker、少量探索和 propensity 日志，防止只观察所选模型造成偏差。

本 Notebook 用 NumPy 构造 small/large 模型结果，手写逻辑路由器、阈值校准、预算分配、IPS 与故障状态机。

In [ ]:
import hashlib, json, math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED114=11401; rng114=np.random.default_rng(SEED114); n114=600  # 计算并保存当前步骤的中间状态。
difficulty114=rng114.uniform(0,1,n114); risk114=(rng114.random(n114)<.15).astype(float); length114=rng114.uniform(0,1,n114); X114=np.c_[np.ones(n114),difficulty114,risk114,length114]  # 计算并保存当前步骤的中间状态。
q_small114=np.clip(.98-.65*difficulty114-.18*risk114+rng114.normal(0,.04,n114),0,1); q_large114=np.clip(.99-.22*difficulty114-.05*risk114+rng114.normal(0,.025,n114),0,1); c_small114,c_large114=.002,.02  # 计算并保存当前步骤的中间状态。
assert X114.shape==(600,4) and q_small114.shape==q_large114.shape==(600,)  # 用受控断言验证关键不变量。
assert q_large114.mean()>q_small114.mean()  # 用受控断言验证关键不变量。
assert c_large114>c_small114 and SEED114==11401  # 用受控断言验证关键不变量。

## 1. 路由不是“难题给大模型”一句话

请求特征只能使用路由时可见信息，如长度、语言、意图、检索置信、风险等级和小模型自评；不能使用事后答案正确性。模型表记录版本、能力、安全边界、上下文、p95、价格和区域。高风险请求可设置强制大模型/人工，而非交给平均效用。

In [ ]:
MODELS114={"small":{"cost":c_small114,"p95":.25,"max_context":8192},"large":{"cost":c_large114,"p95":1.1,"max_context":32768}}  # 计算并保存当前步骤的中间状态。
def feasible114(model,context,slo): return context<=MODELS114[model]["max_context"] and MODELS114[model]["p95"]<=slo  # 定义本节可复用的核心函数。
assert feasible114("small",4000,.5)  # 用受控断言验证关键不变量。
assert not feasible114("small",12000,.5)  # 用受控断言验证关键不变量。
assert not feasible114("large",4000,.5) and feasible114("large",12000,1.5)  # 用受控断言验证关键不变量。

## 2. 用边际质量收益定义离线 Oracle

在 shadow/offline 中让所有候选处理同一 case，得到 `gain=q_large-q_small`。若收益超过额外成本的业务价格，或请求高风险，则升级。这个 oracle 依赖两边都有结果，不能直接从历史被选择模型日志构造无偏标签。

In [ ]:
value_per_quality114=.08; gain114=q_large114-q_small114; oracle_large114=(gain114*value_per_quality114>(c_large114-c_small114))|(risk114==1)  # 计算并保存当前步骤的中间状态。
assert oracle_large114.dtype==bool  # 用受控断言验证关键不变量。
assert oracle_large114[risk114==1].all()  # 用受控断言验证关键不变量。
assert 0<oracle_large114.mean()<1  # 用受控断言验证关键不变量。

## 3. 手写逻辑回归预测是否值得升级

标签是 oracle 的升级决策，特征在请求前可见。训练/测试按时间或用户切分，不能随机泄漏模板族。下面用稳定 sigmoid 和全 batch 梯度下降；实际也可直接回归边际收益，再结合不同成本实时决策。

In [ ]:
def sigmoid114(z):  # 定义本节可复用的核心函数。
    z=np.asarray(z,float); out=np.empty_like(z); pos=z>=0; out[pos]=1/(1+np.exp(-z[pos])); ez=np.exp(z[~pos]); out[~pos]=ez/(1+ez); return out  # 计算并保存当前步骤的中间状态。
train114=np.arange(n114)<450; test114=~train114; w114=np.zeros(X114.shape[1]); y114=oracle_large114.astype(float); losses114=[]  # 计算并保存当前步骤的中间状态。
for _ in range(500):  # 遍历输入元素以累积或检查结果。
    p=sigmoid114(X114[train114]@w114); losses114.append(float(np.mean(-(y114[train114]*np.log(p+1e-12)+(1-y114[train114])*np.log(1-p+1e-12))))); w114-=.4*(X114[train114].T@(p-y114[train114])/train114.sum())  # 计算并保存当前步骤的中间状态。
pred_prob114=sigmoid114(X114[test114]@w114)  # 计算并保存当前步骤的中间状态。
assert losses114[-1]<losses114[0]*.8  # 用受控断言验证关键不变量。
assert np.isfinite(w114).all() and np.all((pred_prob114>=0)&(pred_prob114<=1))  # 用受控断言验证关键不变量。
assert w114[1]>0 and w114[2]>0  # 用受控断言验证关键不变量。

## 4. 阈值由质量门槛和成本预算校准

threshold 越低，更多请求升级，质量/成本/延迟一起上升。不能只最大化路由分类 accuracy；应画 cost-quality frontier，在高风险零漏路由和最低质量下选择最便宜点。阈值锁定后只在 test 报告一次。

In [ ]:
test_idx114=np.where(test114)[0]  # 计算并保存当前步骤的中间状态。
def evaluate_threshold114(t):  # 定义本节可复用的核心函数。
    use_large=(pred_prob114>=t)|(risk114[test114]==1); quality=np.where(use_large,q_large114[test114],q_small114[test114]); cost=np.where(use_large,c_large114,c_small114); return {"t":t,"large_rate":use_large.mean(),"quality":quality.mean(),"cost":cost.mean(),"risk_miss":np.sum((risk114[test114]==1)&~use_large)}  # 计算并保存当前步骤的中间状态。
frontier114=[evaluate_threshold114(t) for t in np.linspace(0,1,21)]; feasible_points114=[r for r in frontier114 if r["quality"]>=.83 and r["risk_miss"]==0]; chosen114=min(feasible_points114,key=lambda r:r["cost"])  # 计算并保存当前步骤的中间状态。
assert chosen114["quality"]>=.83 and chosen114["risk_miss"]==0  # 用受控断言验证关键不变量。
assert c_small114<=chosen114["cost"]<=c_large114  # 用受控断言验证关键不变量。
assert 0<=chosen114["large_rate"]<=1  # 用受控断言验证关键不变量。

## 5. Cascade 可利用小模型输出后的置信信号

先跑 small，再依据校准 confidence、拒答、格式失败或 verifier 决定升级，能看到更强信号但已支付 small 成本并增加尾延迟。confidence 必须在 holdout 校准；高置信错误需要专门 slice，不能把 self-reported confidence 当真。

In [ ]:
small_conf114=np.clip(q_small114+rng114.normal(0,.08,n114),0,1)  # 计算并保存当前步骤的中间状态。
def cascade114(conf,risk,threshold=.72): return (conf<threshold)|(risk==1)  # 定义本节可复用的核心函数。
cascade_large114=cascade114(small_conf114[test114],risk114[test114]); cascade_quality114=np.where(cascade_large114,q_large114[test114],q_small114[test114]); cascade_cost114=c_small114+cascade_large114*c_large114  # 计算并保存当前步骤的中间状态。
assert cascade_large114[risk114[test114]==1].all()  # 用受控断言验证关键不变量。
assert cascade_quality114.mean()>q_small114[test114].mean()  # 用受控断言验证关键不变量。
assert cascade_cost114.mean()>=c_small114  # 用受控断言验证关键不变量。

## 6. 批量预算下按收益/额外成本分配

若每分钟只能升级 B 个请求，可按预测边际收益除以额外成本排序，同时先保留硬风险请求。这里用真实 gain 作为离线 oracle 演示；线上换成校准预测。预算优化必须防止某语言/租户长期只得到弱模型。

In [ ]:
def allocate_budget114(pred_gain,risk,budget):  # 定义本节可复用的核心函数。
    chosen=set(np.where(risk==1)[0].tolist())  # 计算并保存当前步骤的中间状态。
    if len(chosen)>budget: raise ValueError("risk_exceeds_budget")  # 按当前条件选择后续控制路径。
    order=np.argsort(-pred_gain)  # 计算并保存当前步骤的中间状态。
    for i in order:  # 遍历输入元素以累积或检查结果。
        if len(chosen)>=budget: break  # 按当前条件选择后续控制路径。
        chosen.add(int(i))  # 执行当前语句以推进本节示例。
    out=np.zeros(len(pred_gain),bool); out[list(chosen)]=True; return out  # 计算并保存当前步骤的中间状态。
alloc114=allocate_budget114(gain114[test114],risk114[test114],60)  # 计算并保存当前步骤的中间状态。
assert alloc114.sum()==60  # 用受控断言验证关键不变量。
assert alloc114[risk114[test114]==1].all()  # 用受控断言验证关键不变量。
assert np.mean(gain114[test114][alloc114])>np.mean(gain114[test114][~alloc114])  # 用受控断言验证关键不变量。

## 7. 线上反馈有选择偏差，需要探索与 propensity

只知道被路由模型的奖励，若从不让 small 处理难题，就无法估计 small 真实表现。对安全可探索流量随机化一小部分，记录候选集、选择概率和结果，用 IPS/SNIPS 评估新路由；高风险请求不参与随机降级。

In [ ]:
actions114=np.array([0,1,1,0,1]); propensity114=np.array([.8,.2,.2,.8,.2]); reward_obs114=np.array([.7,.95,.8,.6,.9]); target_prob114=np.array([.5,.5,.5,.5,.5]); weights114=target_prob114/propensity114  # 计算并保存当前步骤的中间状态。
ips114=float(np.mean(weights114*reward_obs114)); snips114=float(np.sum(weights114*reward_obs114)/np.sum(weights114))  # 计算并保存当前步骤的中间状态。
assert np.all(propensity114>0)  # 用受控断言验证关键不变量。
assert math.isfinite(ips114) and 0<=snips114<=1  # 用受控断言验证关键不变量。
assert not math.isclose(ips114,snips114)  # 用受控断言验证关键不变量。

## 8. Fallback、熔断、缓存和版本发布

large 超时不一定降级 small：高风险请求可能应失败关闭或人工升级；低风险可 fallback。路由器、候选模型和 grader 分别版本化，shadow/canary 监控质量、成本、p95、各 slice 路由率和供应商故障；熔断后避免重试风暴。

In [ ]:
def failure_policy114(risk,large_health,small_health):  # 定义本节可复用的核心函数。
    if large_health: return "large"  # 按当前条件选择后续控制路径。
    if risk: return "human_or_fail_closed"  # 按当前条件选择后续控制路径。
    if small_health: return "small_fallback"  # 按当前条件选择后续控制路径。
    return "fail"  # 返回当前分支计算出的结果。
manifest114={"schema":1,"router":"logistic-v3","features":["difficulty","risk","length"],"models":{"small":"s-v5","large":"l-v2"},"quality_floor":.83,"exploration":"safe_only","high_risk":"forced_large_or_human"}; digest114=hashlib.sha256(json.dumps(manifest114,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert failure_policy114(1,False,True)=="human_or_fail_closed"  # 用受控断言验证关键不变量。
assert failure_policy114(0,False,True)=="small_fallback" and failure_policy114(0,False,False)=="fail"  # 用受控断言验证关键不变量。
assert len(digest114)==64 and manifest114["quality_floor"]==.83  # 用受控断言验证关键不变量。

## 面试总结

完整回答是：**请求/模型合同 → 全候选 shadow 标签 → 边际收益 oracle → 可用特征路由器 → quality/risk 门槛下调阈值 → cascade/批量预算 → 安全探索与 propensity → fallback/circuit/canary**。路由目标不是猜“哪题难”，而是在硬安全约束下购买最有价值的额外模型能力。

延伸阅读：[RouteLLM](https://arxiv.org/abs/2406.18665)、[FrugalGPT](https://arxiv.org/abs/2305.05176)、[Selective Classification](https://arxiv.org/abs/1705.08500)。